# Project 2

Author: Evan Whitfield

Course: ST554 - Spring 2026

Purpose: Project 2

Necessary imports for Part 1 of the project below.

In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
import Project2 as pro2

The below code was necessary to run instead of having to refresh the kernel after I made changes to the script file.

In [3]:
import importlib

importlib.reload(pro2)

<module 'Project2' from '/home/jupyter-edwhitfi@ncsu.edu/Project2.py'>

## Data Read-in (From CSV)

Setting up the spark session and reading in the data from the .csv file air.csv.

In [4]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

air_data = pro2.SparkDataChecker.read_csv(spark, 'air.csv')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 23:33:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 23:33:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 23:33:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Check-In Range Function Testing

Below we will be testing the various methods in our script, starting with the `check-in-range` method.

In [5]:
air_data.check_in_range("T")

Must give at least one of lower or upper.


`check_in_range` fails if there is not a range to check.

In [6]:
air_data.check_in_range("Date", lower = 11)

Column 'Date' is not numeric.


`check_in_range` fails if the column is not numeric. User must give a column with a numeric type.

In [7]:
air_data.check_in_range("CO(GT)", lower = 2.1).show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|          false|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|  

26/03/26 23:33:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


`check_in_range` shows us when CO(GT) is 2.1 or above. These could potentially be considered "high" CO levels.

In [8]:
air_data.check_in_range("CO(GT)", lower = 1.5, upper = 2.1).show()

26/03/26 23:33:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|          false|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|  

`check_in_range` shows us when CO(GT) is between 1.5 and 2.1. These could be potentially be considered "medium" CO levels.

In [9]:
air_data.check_in_range("CO(GT)",  upper = 1.49).show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|          false|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|          false|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|  

26/03/26 23:33:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


`check_in_range` shows us when CO(GT) is below 1.5. These could be considered potentially "low" CO levels.

In [10]:
air_data.summarize_min_max("CO(GT)", groupedby = 'Date')

,Date,min,max
0,9/2/2004,-200.0,5.2
1,12/26/2004,0.3,3.8
2,2/18/2005,-200.0,4.8
3,10/10/2004,-200.0,3.7
4,10/11/2004,0.4,4.8
...,...,...,...
386,1/23/2005,-200.0,0.8
387,6/28/2004,0.4,5.6
388,8/16/2004,0.3,2.1
389,12/20/2004,0.6,4.1


Here, we grouped the data by 'Date' and looked at the minimum and maximum CO levels for the dates in our dataset. In the future, -200 values should be removed from the consideration as they are considered missing values. As it currently is, there is no way to see the minimum CO reading for 9/2/2004 because there was a missing CO reading from that date that is being shown as the minimum value.

In [11]:
air_data.summarize_min_max(groupedby = 'Date')

26/03/26 23:34:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/26 23:34:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date
 Schema: _c0, Date
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


,Date,_c0_min,_c0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,9/2/2004,4206,4229,-200.0,5.2,832,1464,-200,-200,3.1,...,1312,2366,738,1738,21.0,38.7,17.3,57.2,1.1729,1.5165
1,12/26/2004,6966,6989,0.3,3.8,848,1423,-200,-200,0.7,...,1062,1531,438,1381,10.4,14.9,57.9,82.9,0.9640,1.1381
2,2/18/2005,8262,8285,-200.0,4.8,872,1288,-200,-200,0.7,...,794,1333,350,1502,5.4,11.8,30.9,52.9,0.4223,0.4922
3,10/10/2004,5118,5141,-200.0,3.7,941,1520,-200,-200,3.6,...,1406,1812,549,1357,19.4,24.8,56.9,80.5,1.6507,1.8881
4,10/11/2004,5142,5165,0.4,4.8,825,1464,-200,-200,1.4,...,1302,2072,494,1625,17.9,23.2,52.0,77.7,1.3032,1.6384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,1/23/2005,7638,7661,-200.0,0.8,849,1174,-200,-200,1.6,...,810,1220,737,1445,2.5,8.9,36.0,78.2,0.3861,0.7074
387,6/28/2004,2622,2645,0.4,5.6,827,1441,-200,-200,2.7,...,1438,2568,619,1791,22.8,40.4,19.0,50.1,1.3447,1.4854
388,8/16/2004,3798,3821,0.3,2.1,749,1028,-200,-200,1.5,...,1244,1624,458,1135,21.3,39.7,14.6,47.0,0.9295,1.3722
389,12/20/2004,6822,6845,0.6,4.1,762,1185,-200,-200,0.7,...,783,1250,454,1104,6.8,9.8,32.4,82.8,0.3340,0.8561


Again, we grouped by Date, but without a given numeric column, the program returns a min and a max for every numeric column. Again, any missing value is set to -200, which appears in the min category. Data cleaning is necessary to return the proper minimum value recorded for the given Date.

In [12]:
air_data.summarize_counts("Date")

,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


Here, we wanted to see if the date column was recognized as a string column, and if so, how many recordings were made for each date. It appears that a recording was made every hour of each day.

In [13]:
air_data.summarize_counts("Date", "T")

'T' is not a string column.


,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


In [14]:
air_data.summarize_counts("T", "Date")

'T' is not a string column.


,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


In [15]:
air_data.summarize_counts("T")

'T' is not a string column.


We wanted to run summarize_counts with column "T" a few times to verify the correct results. Since "T" is for temperature (and should be numeric), the method tells us that the column is not a string column, and therefore cannot be used in this particular method.

## Reading in Data from pandas.

Below we read in the data via and a pandas dataframe. We then verified that the `check_in_range` method still worked properly.

In [17]:
df = pd.read_csv('air.csv')
air_data2 = pro2.SparkDataChecker.from_pandas(spark, df)

air_data2.check_in_range("T", lower = 10, upper = 12).show()

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_in_range|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|     false|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|     false|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|      true|
|         

The data was read into the class correctly. The method shows us when the temperature is between 10 and 12. I believe that these are Celsius temperature values. I was looking for when the temperature might be near freezing, but the column would have returned false for all responses. This would not have verified the method worked, so I went for the range of 10 to 12 instead. 